# 🏆 Nismara Transport — Laporan Kinerja Staff & KPI Bulanan

Notebook ini menghitung kinerja seluruh **Staff / Manager** secara akurat mengikuti formula KPI resmi di `nismara-logistics/lib/managerSalary.ts` (dengan aturan Anti-Abuse + zona waktu WIB `Asia/Jakarta`), ditambah metrik **Respon Rekrutmen (`applications` & `registrations`)** serta status klaim gaji (`managersalaryrecords`).

Seluruh hasil otomatis diekspor ke subfolder **`report/YYYY-MM/`**:
- `staff_kpi_<bulan>_<tahun>.csv`
- `staff_kpi_<bulan>_<tahun>.json`
- `laporan_kpi_staff_<bulan>_<tahun>.md`

In [ ]:
import os
import json
import datetime
import pandas as pd
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()
MONGODB_URI = os.getenv("MONGODB_URI", "mongodb://localhost:27017/nismara_db")
GUILD_ID = os.getenv("DISCORD_GUILD_ID", "863959415702028318")

MONTH_NAMES = [
    "Januari", "Februari", "Maret", "April", "Mei", "Juni",
    "Juli", "Agustus", "September", "Oktober", "November", "Desember"
]

# =============================================================================
# ⚙️ PARAMETER PERIODE KPI STAFF (WIB / Asia/Jakarta)
# =============================================================================
BULAN = 10         # 1 = Januari ... 10 = Oktober, 11 = November
TAHUN = 2026       # Tahun evaluasi
# =============================================================================

NAMA_BULAN = MONTH_NAMES[BULAN - 1]
MONTH_KEY = f"{TAHUN}-{BULAN:02d}"
REPORT_DIR = os.path.join("report", MONTH_KEY)
os.makedirs(REPORT_DIR, exist_ok=True)

# Rentang bulan kalender penuh dalam WIB (UTC+7 -> kurangi 7 jam dari UTC, identik dengan lib/managerSalary.ts)
START_WIB_UTC = datetime.datetime(TAHUN, BULAN, 1, 0, 0, 0, tzinfo=datetime.timezone.utc) - datetime.timedelta(hours=7)
if BULAN == 12:
    END_WIB_UTC = datetime.datetime(TAHUN + 1, 1, 1, 0, 0, 0, tzinfo=datetime.timezone.utc) - datetime.timedelta(hours=7)
else:
    END_WIB_UTC = datetime.datetime(TAHUN, BULAN + 1, 1, 0, 0, 0, tzinfo=datetime.timezone.utc) - datetime.timedelta(hours=7)

print(f"📅 Periode KPI Staff : {NAMA_BULAN} {TAHUN} ({MONTH_KEY})")
print(f"🕒 Rentang UTC (WIB) : {START_WIB_UTC.isoformat()} s/d {END_WIB_UTC.isoformat()}")
print(f"📁 Output Folder     : {REPORT_DIR}")

In [ ]:
client = MongoClient(MONGODB_URI)
db = client.get_default_database()

# 1. Identifikasi seluruh Staff / Manager aktif & yang tercatat pada periode ini
staff_users = list(db["users"].find({
    "$or": [
        {"discordRole": {"$in": ["manager", "admin"]}},
        {"truckyRole": {"$regex": "manajer|manager|owner|hr|staff|director|deputy", "$options": "i"}}
    ]
}, {
    "discordId": 1, "name": 1, "truckyId": 1, "discordRole": 1, "truckyRole": 1, "topManager": 1
}))

salary_records = list(db["managersalaryrecords"].find({"month": MONTH_KEY}))
salary_by_id = {str(r.get("managerId")): r for r in salary_records if r.get("managerId")}

staff_map = {}
for u in staff_users:
    did = str(u.get("discordId") or "")
    if did:
        staff_map[did] = u

for did in salary_by_id.keys():
    if did not in staff_map:
        u = db["users"].find_one({"discordId": did}) or {"discordId": did, "name": f"Staff {did}"}
        staff_map[did] = u

print(f"👥 Total Staff/Manager terdeteksi: {len(staff_map)}")

In [ ]:
# 2. Kalkulasi seluruh 10 parameter KPI resmi (lib/managerSalary.ts) + Respon Rekrutmen (applications & registrations)
kpi_rows = []

for manager_id, u in staff_map.items():
    # 1. Tiket Support (1 pt/tiket, anti-abuse: bukan tiket sendiri)
    tickets_handled = db["tickets"].count_documents({
        "managerId": manager_id,
        "status": {"$in": ["resolved", "closed"]},
        "discordId": {"$ne": manager_id},
        "updatedAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
    })

    # 2. Order Pembelian Fleet (1 pt/order)
    fleet_orders = db["fleetorders"].count_documents({
        "managerId": manager_id,
        "status": "completed",
        "buyerId": {"$ne": manager_id},
        "updatedAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
    })

    # 3. Konfirmasi Servis Armada (1 pt/servis)
    fleet_services = db["fleetmaintenanceorders"].count_documents({
        "managerId": manager_id,
        "status": {"$in": ["completed", "in_service"]},
        "discordId": {"$ne": manager_id},
        "$or": [
            {"maintenanceStartAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC}},
            {"updatedAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC}},
        ],
    })

    # 4. Penyelenggaraan Convoy (2 pt/convoy + 1 pt/peserta luar)
    convoys = list(db["convoylobby"].find({
        "setBy": manager_id,
        "startDate": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
    }))
    convoys_hosted = len(convoys)
    convoy_participants = 0
    for c in convoys:
        for p in (c.get("partisipan") or []):
            if p and p.get("discordId") and str(p.get("discordId")) != manager_id:
                convoy_participants += 1
    convoy_points = convoys_hosted * 2 + convoy_participants

    # 5. Special Contract (5 pt/kontrak)
    contracts_created = db["contracts"].count_documents({
        "setBy": manager_id,
        "$or": [
            {"startDate": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC}},
            {"setAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC}},
            {"createdAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC}},
        ],
    })
    contract_points = contracts_created * 5

    # 6. Event NC Boost (5 pt/event)
    boost_events = db["ncevents"].count_documents({
        "setBy": manager_id,
        "$or": [
            {"startDate": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC}},
            {"setAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC}},
            {"createdAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC}},
        ],
    })
    boost_points = boost_events * 5

    # 7. Review Mod Market (2 pt/mod)
    mods_reviewed = db["marketitems"].count_documents({
        "reviewerId": manager_id,
        "status": {"$in": ["approved", "rejected"]},
        "sellerId": {"$ne": manager_id},
        "updatedAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
    })
    mod_points = mods_reviewed * 2

    # 8. Jarak Tempuh Mengemudi (1 pt / 1.000 km)
    job_agg = list(db["jobhistories"].aggregate([
        {
            "$match": {
                "driverId": manager_id,
                "jobStatus": "COMPLETED",
                "cancelPenaltyApplied": {"$ne": True},
                "completedAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
            }
        },
        {"$group": {"_id": None, "totalKm": {"$sum": "$distanceKm"}, "totalJobs": {"$sum": 1}}},
    ]))
    distance_km = round(job_agg[0]["totalKm"], 2) if job_agg else 0.0
    total_jobs = job_agg[0]["totalJobs"] if job_agg else 0
    distance_points = int(distance_km // 1000)

    # 9. Promosi / Kelulusan Intern (3 pt/kelulusan)
    intern_promotions = db["internpromotions"].count_documents({
        "managerId": manager_id,
        "internDiscordId": {"$ne": manager_id},
        "promotedAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
    })
    intern_promo_points = intern_promotions * 3

    # 10. Evaluasi / Audit Intern (2 pt/evaluasi)
    intern_evals = db["driverevaluations"].count_documents({
        "closedByManagerId": manager_id,
        "status": "closed",
        "driverId": {"$ne": manager_id},
        "closedAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
    })
    intern_eval_points = intern_evals * 2

    # 11. Metrik Tambahan HR: Respon Pendaftaran & Rekrutmen (applications & registrations)
    apps_handled = db["applications"].count_documents({
        "claimedBy.discordId": manager_id,
        "updatedAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
    })
    apps_accepted = db["applications"].count_documents({
        "claimedBy.discordId": manager_id,
        "status": "Accepted",
        "updatedAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
    })
    apps_rejected = db["applications"].count_documents({
        "claimedBy.discordId": manager_id,
        "status": "Rejected",
        "updatedAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
    })
    regs_handled = db["registrations"].count_documents({
        "managerId": manager_id,
        "updatedAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
    })

    # 12. Tindakan Kedisiplinan (pointhistories yang diproses manager)
    penalty_actions = db["pointhistories"].count_documents({
        "managerId": manager_id,
        "createdAt": {"$gte": START_WIB_UTC, "$lt": END_WIB_UTC},
    })

    total_kpi_points = (
        tickets_handled + fleet_orders + fleet_services + convoy_points +
        contract_points + boost_points + mod_points + distance_points +
        intern_promo_points + intern_eval_points
    )

    sal = salary_by_id.get(manager_id)
    payroll_status = sal.get("status", "UNCLAIMED") if sal else "UNCLAIMED"
    nc_granted = (sal.get("rewardsGranted") or {}).get("ncAmount", 0) if sal else 0

    kpi_rows.append({
        "managerId": manager_id,
        "name": u.get("name") or f"Staff {manager_id}",
        "truckyRole": u.get("truckyRole") or u.get("discordRole") or "manager",
        "totalKpiPoints": total_kpi_points,
        "ticketsHandled": tickets_handled,
        "recruitmentHandled": apps_handled + regs_handled,
        "appsAccepted": apps_accepted,
        "appsRejected": apps_rejected,
        "fleetOrdersHandled": fleet_orders,
        "fleetServicesHandled": fleet_services,
        "convoysHosted": convoys_hosted,
        "convoyParticipants": convoy_participants,
        "contractsCreated": contracts_created,
        "boostEventsCreated": boost_events,
        "modsReviewed": mods_reviewed,
        "internPromotions": intern_promotions,
        "internEvaluations": intern_evals,
        "penaltyActions": penalty_actions,
        "distanceKm": distance_km,
        "totalJobs": total_jobs,
        "distancePoints": distance_points,
        "payrollStatus": payroll_status,
        "payrollNcGranted": nc_granted,
    })

df_kpi = pd.DataFrame(kpi_rows).sort_values(by=["totalKpiPoints", "recruitmentHandled"], ascending=False).reset_index(drop=True)
display(df_kpi)

In [ ]:
# 3. Export CSV, JSON, dan Laporan Markdown ke report/YYYY-MM/
csv_out = os.path.join(REPORT_DIR, f"staff_kpi_{NAMA_BULAN.lower()}_{TAHUN}.csv")
json_out = os.path.join(REPORT_DIR, f"staff_kpi_{NAMA_BULAN.lower()}_{TAHUN}.json")
md_out = os.path.join(REPORT_DIR, f"laporan_kpi_staff_{NAMA_BULAN.lower()}_{TAHUN}.md")

df_kpi.to_csv(csv_out, index=False, encoding="utf-8-sig")

summary_payload = {
    "period": {"monthKey": MONTH_KEY, "month": BULAN, "year": TAHUN, "label": f"{NAMA_BULAN} {TAHUN}"},
    "generatedAt": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "teamTotals": {
        "totalStaff": int(len(df_kpi)),
        "activeStaffWithPoints": int((df_kpi["totalKpiPoints"] > 0).sum()) if not df_kpi.empty else 0,
        "totalKpiPoints": int(df_kpi["totalKpiPoints"].sum()) if not df_kpi.empty else 0,
        "avgKpiPoints": round(float(df_kpi["totalKpiPoints"].mean()), 2) if not df_kpi.empty else 0,
        "ticketsHandled": int(df_kpi["ticketsHandled"].sum()) if not df_kpi.empty else 0,
        "recruitmentHandled": int(df_kpi["recruitmentHandled"].sum()) if not df_kpi.empty else 0,
        "appsAccepted": int(df_kpi["appsAccepted"].sum()) if not df_kpi.empty else 0,
        "appsRejected": int(df_kpi["appsRejected"].sum()) if not df_kpi.empty else 0,
        "fleetOrdersHandled": int(df_kpi["fleetOrdersHandled"].sum()) if not df_kpi.empty else 0,
        "fleetServicesHandled": int(df_kpi["fleetServicesHandled"].sum()) if not df_kpi.empty else 0,
        "convoysHosted": int(df_kpi["convoysHosted"].sum()) if not df_kpi.empty else 0,
        "contractsCreated": int(df_kpi["contractsCreated"].sum()) if not df_kpi.empty else 0,
        "boostEventsCreated": int(df_kpi["boostEventsCreated"].sum()) if not df_kpi.empty else 0,
        "modsReviewed": int(df_kpi["modsReviewed"].sum()) if not df_kpi.empty else 0,
        "internPromotions": int(df_kpi["internPromotions"].sum()) if not df_kpi.empty else 0,
        "internEvaluations": int(df_kpi["internEvaluations"].sum()) if not df_kpi.empty else 0,
        "totalDistanceKm": round(float(df_kpi["distanceKm"].sum()), 2) if not df_kpi.empty else 0,
    },
    "staff": df_kpi.to_dict(orient="records"),
}

with open(json_out, "w", encoding="utf-8") as f:
    json.dump(summary_payload, f, ensure_ascii=False, indent=2)

t = summary_payload["teamTotals"]
lines = [
    f"# 🏆 Laporan Kinerja Staff & KPI — {NAMA_BULAN} {TAHUN}",
    f"**Periode:** {MONTH_KEY} (Zona Waktu WIB / Asia/Jakarta)  ",
    f"**Total Staff Terdata:** {t['totalStaff']} ({t['activeStaffWithPoints']} aktif mencetak poin KPI)  ",
    "",
    "## 📊 Ringkasan Kinerja Tim",
    f"- **Total Poin KPI Tim:** {t['totalKpiPoints']:,} poin (Rata-rata: {t['avgKpiPoints']} poin/staff)",
    f"- **Tiket Support Diselesaikan:** {t['ticketsHandled']:,} tiket",
    f"- **Pendaftaran / Rekrutmen Ditangani:** {t['recruitmentHandled']:,} (Diterima: {t['appsAccepted']}, Ditolak: {t['appsRejected']})",
    f"- **Order & Servis Armada:** {t['fleetOrdersHandled']:,} order selesai | {t['fleetServicesHandled']:,} servis",
    f"- **Event & Komunitas:** {t['convoysHosted']:,} convoy | {t['contractsCreated']:,} special contract | {t['boostEventsCreated']:,} NC boost",
    f"- **Moderasi & Pembinaan Intern:** {t['modsReviewed']:,} mod direview | {t['internEvaluations']:,} evaluasi | {t['internPromotions']:,} promosi",
    f"- **Total Jarak Tempuh Staff:** {t['totalDistanceKm']:,.2f} km",
    "",
    "## 📋 Papan Peringkat KPI Staff",
    "| Rank | Nama Staff | Role | Poin KPI | Tiket | Rekrutmen | Fleet (Order/Servis) | Event (Convoy/SC/Boost) | Intern (Eval/Promo) | Mod | KM | Payroll |",
    "| :--- | :--- | :--- | ---: | ---: | ---: | :---: | :---: | :---: | ---: | ---: | :---: |",
]

for idx, row in df_kpi.iterrows():
    lines.append(
        f"| {idx + 1} | **{row['name']}** | {row['truckyRole']} | **{row['totalKpiPoints']}** | "
        f"{row['ticketsHandled']} | {row['recruitmentHandled']} | "
        f"{row['fleetOrdersHandled']}/{row['fleetServicesHandled']} | "
        f"{row['convoysHosted']}/{row['contractsCreated']}/{row['boostEventsCreated']} | "
        f"{row['internEvaluations']}/{row['internPromotions']} | "
        f"{row['modsReviewed']} | {row['distanceKm']:,.0f} | {row['payrollStatus']} |"
    )

with open(md_out, "w", encoding="utf-8") as f:
    f.write("
".join(lines) + "
")

print(f"✅ CSV disimpan ke : {csv_out}")
print(f"✅ JSON disimpan ke: {json_out}")
print(f"✅ MD disimpan ke  : {md_out}")